Need to run this in web version instead of VSCode. smth somewhere my python is messed up.

## Environment Setup & Hamiltonian Definition

Import all required libraries

### Physical Background
A superconducting transmon qubit is a macroscopic quantum circuit object that behaves like an anharmonic oscillator. When truncated to its two lowest energy levels, it is mathematically equivalent to a spin-1/2 system. In the **rotating frame** (co-rotating with the microwave drive frequency), the effective Hamiltonian is:

$$H(t) = \underbrace{\frac{\delta}{2}Z}_{\text{drift (detuning)}} + \underbrace{\frac{\Omega(t)}{2}X}_{\text{microwave drive}}$$

where $\delta = \omega_q - \omega_d$ is the qubit-drive detuning and $\Omega(t)$ is the time-dependent pulse envelope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.proj3d import proj_transform
import warnings
warnings.filterwarnings('ignore')

from qiskit_dynamics import Solver
from qiskit_dynamics.signals import Signal        
from qiskit.quantum_info import Statevector       

from scipy.optimize import minimize

print("Imports good")
print(f"  numpy   {np.__version__}")
import scipy; print(f"  scipy   {scipy.__version__}")
import qiskit; print(f"  qiskit  {qiskit.__version__}")
import qiskit_dynamics; print(f"  qiskit-dynamics  {qiskit_dynamics.__version__}")

In [ ]:
# Pauli Matrices

# Z = |0><0| - |1><1| : qubit energy observable (population difference)
# X = |0><1| + |1><0| : bit-flip operator (drives transitions between |0> and |1>)
# Y = -i|0><1| + i|1><0| : orthogonal rotation axis

sigma_x = np.array([[0, 1],
                     [1, 0]], dtype=complex)   # Pauli-X

sigma_y = np.array([[0, -1j],
                     [1j,  0]], dtype=complex)  # Pauli-Y

sigma_z = np.array([[1,  0],
                     [0, -1]], dtype=complex)   # Pauli-Z (computational basis)

# Physical system parameters (standardized)
T_gate   = 20.0          
delta_0  = 0.05          

#  Hamiltonians 
H_drift = 0.5 * delta_0 * sigma_z        

H_ctrl_op = 0.5 * sigma_x               

print("Drift Hamiltonian H₀ = (δ/2)·Z:")
print(H_drift)
print()
print("Control Hamiltonian operator H₁ = (1/2)·X:")
print(H_ctrl_op)
print()
print(f"System parameters: T_gate={T_gate} ns,  δ₀={delta_0} rad/ns")

---
## Creating the Ideal Gaussian X Gate

### Gaussian Pulse Design
The qubit is driven via a Gaussian pulse envelope:

$$\Omega(t) = A \cdot \exp\!\left(-\frac{(t - t_0)^2}{2\sigma^2}\right)$$

For a perfect $\pi$-rotation (Pauli-X gate: $|0\rangle \rightarrow |1\rangle$) the area theorem demands:

$$\int_{-\infty}^{+\infty} \frac{\Omega(t)}{2}\, dt = \frac{\pi}{2}$$

Since $\int_{-\infty}^{+\infty} e^{-(t-t_0)^2/(2\sigma^2)}\,dt = \sigma\sqrt{2\pi}$, the required amplitude is:

$$\boxed{A = \frac{\pi}{\sigma\sqrt{2\pi}}}$$

I'll simulate the pulse without any environmental noise to establish the perfect baseline trajectory.

In [ ]:
# Ideal parameters
sigma_ideal = T_gate / 6          
t0          = T_gate / 2          

# Set up ideal area integral
A_ideal = np.pi / (sigma_ideal * np.sqrt(2 * np.pi))

print(f"Ideal Gaussian pulse parameters:")
print(f"  Centre t₀   = {t0:.2f} ns")
print(f"  Width  σ    = {sigma_ideal:.4f} ns")
print(f"  Amplitude A = {A_ideal:.6f} rad/ns")
print(f"  Pulse area check ∫Ω dt = A·σ·√(2π) = {A_ideal * sigma_ideal * np.sqrt(2*np.pi):.6f}  (should be π = {np.pi:.6f})")

#  Optimization & timegrid
t_eval = np.linspace(0, T_gate, 400)    # 400 evaluation points for smooth plots

In [ ]:
# Extract Bloch sphere coordinates from a pure state |ψ⟩

def bloch_vector(sv_array):
    """Return (Bx, By, Bz) Bloch components for a length-2 complex statevector."""
    sv  = np.asarray(sv_array, dtype=complex)
    rho = np.outer(sv, sv.conj())                        # density matrix ρ = |ψ><ψ|
    bx  = float(np.real(np.trace(rho @ sigma_x)))       # ⟨X⟩
    by  = float(np.real(np.trace(rho @ sigma_y)))       # ⟨Y⟩
    bz  = float(np.real(np.trace(rho @ sigma_z)))       # ⟨Z⟩
    return bx, by, bz


def compute_fidelity(sv_final, target=None):
    """State fidelity F = |⟨target|ψ_final⟩|² against |1⟩ by default."""
    if target is None:
        target = np.array([0.0, 1.0], dtype=complex)    # |1⟩ = [0, 1]ᵀ
    sv = np.asarray(sv_final, dtype=complex)
    return float(abs(np.dot(target.conj(), sv))**2)


print("Helper functions defined: bloch_vector(), compute_fidelity()")

In [ ]:
# Ideal simulation (no noise, δ=0) We deliberately set H_drift = 0 here to isolate the ideal pulse behaviour. The rotating frame removes H₀; what remains is pure X-rotation.

def run_simulation(A, sigma, delta=0.0, t_eval=t_eval):
    """
    Solve Schrödinger's equation for one Gaussian pulse.

    Parameters
    ----------
    A      : float  — pulse amplitude [rad/ns]
    sigma  : float  — pulse width [ns]
    delta  : float  — detuning added to H_drift [rad/ns]
    t_eval : array  — time points at which to record the state

    Returns
    -------
    list of Statevector objects, one per t_eval point
    """
    H_static = 0.5 * delta * sigma_z           

    # Qiskit Dynamics Signal: envelope × exp(i·carrier_freq·t)
    pulse_signal = Signal(
        envelope=lambda t, _A=A, _s=sigma: _A * np.exp(-(t - t0)**2 / (2 * _s**2)),
        carrier_freq=0.0
    )

    # Solver proper
    solver = Solver(
        static_hamiltonian=H_static,
        hamiltonian_operators=[H_ctrl_op],
        validate=False,                        # skips Hermitian check for speed if unecessary
    )

    # Solve from t=0 to t=T_gate, recording at t_eval points
    result = solver.solve(
        t_span=[0.0, T_gate],
        y0=Statevector([1.0, 0.0]),          
        signals=[pulse_signal],
        t_eval=t_eval,
    )
    return result.y   # list of Statevector


# Ideal simulation
states_ideal = run_simulation(A_ideal, sigma_ideal, delta=0.0)
sv_final_ideal = np.array(states_ideal[-1])

print(f"Ideal final state |ψ(T)⟩ = {sv_final_ideal}")
print(f"Ideal Gate Fidelity F = {compute_fidelity(sv_final_ideal):.8f}  (target: 1.000000)")

In [ ]:
# Extract trajectory data
def extract_trajectory(states):
    """Convert list of Statevectors into probability and Bloch arrays."""
    prob_0 = np.array([abs(np.array(s)[0])**2 for s in states])
    prob_1 = np.array([abs(np.array(s)[1])**2 for s in states])
    bloch   = np.array([bloch_vector(np.array(s)) for s in states])
    return prob_0, prob_1, bloch

prob0_ideal, prob1_ideal, bloch_ideal = extract_trajectory(states_ideal)
# PLOTS
# P1: Ideal pulse shape + state probabilities 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Phase 2 — Ideal Gaussian π-Pulse: Perfect X-Gate", fontsize=14, fontweight='bold')

# Left: pulse envelope
ax1 = axes[0]
pulse_envelope = A_ideal * np.exp(-(t_eval - t0)**2 / (2 * sigma_ideal**2))
ax1.fill_between(t_eval, pulse_envelope, alpha=0.25, color='steelblue')
ax1.plot(t_eval, pulse_envelope, color='steelblue', lw=2.5, label=f'Gaussian (A={A_ideal:.3f})')
ax1.axhline(0, color='grey', lw=0.8, ls='--')
ax1.set_xlabel("Time [ns]", fontsize=12)
ax1.set_ylabel("Drive Amplitude Ω(t) [rad/ns]", fontsize=12)
ax1.set_title("Ideal Gaussian Pulse Envelope", fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.annotate(f"Area = π (π-rotation)", xy=(t0, A_ideal*0.5),
             ha='center', fontsize=11, color='navy',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='aliceblue', edgecolor='steelblue'))

# Right: state probabilities over time
ax2 = axes[1]
ax2.plot(t_eval, prob0_ideal, color='royalblue', lw=2.5, label='P(|0⟩)')
ax2.plot(t_eval, prob1_ideal, color='tomato',    lw=2.5, label='P(|1⟩)')
ax2.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.6)
ax2.fill_between(t_eval, prob1_ideal, alpha=0.12, color='tomato')
ax2.set_xlabel("Time [ns]", fontsize=12)
ax2.set_ylabel("Population", fontsize=12)
ax2.set_title("Qubit State Probabilities (|0⟩ → |1⟩)", fontsize=12)
ax2.set_ylim(-0.05, 1.08)
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)
ax2.text(T_gate * 0.7, 0.93, f'F = {compute_fidelity(sv_final_ideal):.6f}',
         fontsize=12, color='darkgreen',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='honeydew', edgecolor='green'))

plt.tight_layout()
plt.savefig('phase2_ideal_pulse.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: phase2_ideal_pulse.png")

In [ ]:
# P2: Bloch sphere trajectory (3D)

fig = plt.figure(figsize=(9, 9))
ax  = fig.add_subplot(111, projection='3d')

# Unit sphere wireframe
u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0, np.pi, 60)
sx = np.outer(np.cos(u), np.sin(v))
sy = np.outer(np.sin(u), np.sin(v))
sz = np.outer(np.ones(60), np.cos(v))
ax.plot_wireframe(sx, sy, sz, color='lightgrey', alpha=0.18, linewidth=0.4)

# Axes
for vec, lbl, col in [([1,0,0], '+X', 'red'), ([0,1,0], '+Y', 'green'), ([0,0,1], '+Z|0⟩', 'blue')]:
    ax.quiver(0,0,0,*vec, length=1.25, color=col, arrow_length_ratio=0.08, linewidth=1.8)
    ax.text(vec[0]*1.35, vec[1]*1.35, vec[2]*1.35, lbl, fontsize=11, color=col)

# Trajectory coloured by time (warm=early, cool=late)
bx, by, bz = bloch_ideal[:,0], bloch_ideal[:,1], bloch_ideal[:,2]
for i in range(len(bx)-1):
    frac = i / len(bx)
    col  = plt.cm.plasma(frac)
    ax.plot(bx[i:i+2], by[i:i+2], bz[i:i+2], color=col, lw=2.5)

ax.scatter(*[0], *[0], *[1],  s=120, c='lime',   zorder=5, label='|0⟩ start')
ax.scatter(*bloch_ideal[-1],  s=120, c='magenta', zorder=5, label='|1⟩ end')
ax.scatter(*[0], *[0], *[-1], s=80,  c='magenta', marker='*', alpha=0.5, label='|1⟩ target')

ax.set_xlim([-1.3, 1.3]); ax.set_ylim([-1.3, 1.3]); ax.set_zlim([-1.3, 1.3])
ax.set_xlabel('X', fontsize=11); ax.set_ylabel('Y', fontsize=11); ax.set_zlabel('Z', fontsize=11)
ax.set_title("Ideal π-Pulse Trajectory on Bloch Sphere\n(|0⟩ → |1⟩ via X-rotation)", fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)

sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, T_gate))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.08)
cbar.set_label('Time [ns]', fontsize=11)

plt.tight_layout()
plt.savefig('phase2_bloch_ideal.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: phase2_bloch_ideal.png")

---
## Noise Injection & Chaos Simulation

### Real-World Hardware Imperfections

Physical quantum processors suffer from multiple noise sources simultaneously:

| Noise Type | Physical Origin | Mathematical Model |
|---|---|---|
| Amplitude Jitter | AWG (arbitrary waveform generator) voltage noise, amplifier instability | $A_{\text{noisy}} = A(1 + \epsilon_A)$, $\epsilon_A \sim \mathcal{N}(0, \sigma_A^2)$ |
| Phase Drift / Detuning | Temperature fluctuations shifting qubit frequency, LO phase noise | $H_{\text{drift}} = \frac{\delta}{2}Z$, $\delta \neq 0$ |

Together, these cause the qubit state to **miss** the target $|1\rangle$ — a geometrical error on the Bloch sphere.

In [ ]:
# Fixed noise parameters
np.random.seed(42)              

AMP_JITTER_FRAC = 0.15           
DELTA_NOISE     = delta_0        

print("Noise model parameters:")
print(f"  Amplitude jitter:  +{AMP_JITTER_FRAC*100:.0f}% of ideal amplitude A")
print(f"  Detuning drift  :  δ = {DELTA_NOISE:.4f} rad/ns  →  H_drift = (δ/2)·Z")


def run_noisy_simulation(A_param, sigma_param, amp_jitter=AMP_JITTER_FRAC,
                          delta=DELTA_NOISE, t_eval=t_eval):
    """
    Simulate the qubit evolution under a noisy Gaussian pulse.

    Noise sources applied:
      1. Amplitude jitter : A_effective = A_param × (1 + amp_jitter)
         Simulates over/under-drive from classical control electronics.
      2. Detuning drift   : H_drift = (delta/2)·Z
         Simulates residual qubit frequency offset (environmental decoherence proxy).

    Returns
    -------
    states     : list of Statevectors at each t_eval point
    A_effective: the actually applied amplitude (for bookkeeping)
    """
    A_effective = A_param * (1.0 + amp_jitter)     

    # Noisy drift Hamiltonian; nonzero detuning causes unwanted Z-axis precession
    H_noisy_drift = 0.5 * delta * sigma_z

    pulse_signal = Signal(
        envelope=lambda t, _A=A_effective, _s=sigma_param:
                 _A * np.exp(-(t - t0)**2 / (2 * _s**2)),
        carrier_freq=0.0
    )

    solver = Solver(
        static_hamiltonian=H_noisy_drift,          # with drift active
        hamiltonian_operators=[H_ctrl_op],
        validate=False,
    )

    result = solver.solve(
        t_span=[0.0, T_gate],
        y0=Statevector([1.0, 0.0]),
        signals=[pulse_signal],
        t_eval=t_eval,
    )
    return result.y, A_effective


#  Noisy simulation with ideal parameters
states_noisy, A_eff_noisy = run_noisy_simulation(A_ideal, sigma_ideal)
sv_final_noisy = np.array(states_noisy[-1])
fidelity_noisy = compute_fidelity(sv_final_noisy)

print(f"\nNoisy simulation results:")
print(f"  Intended amplitude : {A_ideal:.6f} rad/ns")
print(f"  Applied amplitude  : {A_eff_noisy:.6f} rad/ns  (+{AMP_JITTER_FRAC*100:.0f}% jitter)")
print(f"  Detuning δ         : {DELTA_NOISE:.4f} rad/ns")
print(f"  Final state |ψ⟩    : {sv_final_noisy}")
print(f"  Gate Fidelity F    : {fidelity_noisy:.6f}   ← degraded by noise")

In [ ]:
# Extract noisy trajectory 
prob0_noisy, prob1_noisy, bloch_noisy = extract_trajectory(states_noisy)

# Figure: Pulse comparison + probability degradation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Noise Injection: How Hardware Errors Corrupt the Gate",
             fontsize=14, fontweight='bold')

# Left: overlaid pulse shapes
ax1 = axes[0]
envelope_ideal = A_ideal * np.exp(-(t_eval - t0)**2 / (2 * sigma_ideal**2))
envelope_noisy = A_eff_noisy * np.exp(-(t_eval - t0)**2 / (2 * sigma_ideal**2))

ax1.fill_between(t_eval, envelope_noisy, alpha=0.15, color='crimson')
ax1.plot(t_eval, envelope_ideal, color='steelblue', lw=2.5, ls='--', label=f'Ideal  A={A_ideal:.3f}')
ax1.plot(t_eval, envelope_noisy, color='crimson',   lw=2.5,         label=f'Noisy  A={A_eff_noisy:.3f}  (+{AMP_JITTER_FRAC*100:.0f}%)')
ax1.set_xlabel("Time [ns]", fontsize=12)
ax1.set_ylabel("Drive Amplitude Ω(t) [rad/ns]", fontsize=12)
ax1.set_title("Ideal vs Noise-Corrupted Pulse", fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)

# Right: probability comparison
ax2 = axes[1]
ax2.plot(t_eval, prob1_ideal, color='steelblue', lw=2.5, ls='--', label=f'Ideal P(|1⟩) → F={compute_fidelity(sv_final_ideal):.4f}')
ax2.plot(t_eval, prob1_noisy, color='crimson',   lw=2.5,         label=f'Noisy P(|1⟩) → F={fidelity_noisy:.4f}')
ax2.axhline(1.0, color='grey', lw=0.8, ls=':', alpha=0.6)
ax2.fill_between(t_eval, prob1_ideal, prob1_noisy, alpha=0.15, color='crimson', label='Fidelity loss region')
ax2.set_xlabel("Time [ns]", fontsize=12)
ax2.set_ylabel("P(|1⟩) — Target State Population", fontsize=12)
ax2.set_title("State Population: Ideal vs Noisy", fontsize=12)
ax2.set_ylim(-0.05, 1.08)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

fidelity_drop = compute_fidelity(sv_final_ideal) - fidelity_noisy
ax2.text(T_gate*0.03, 0.88,
         f'ΔF = {fidelity_drop:.4f} lost to noise!',
         fontsize=11, color='crimson',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#fff0f0', edgecolor='crimson'))

plt.tight_layout()
plt.savefig('phase3_noise_impact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ideal vs noisy bloch trajectories side by side
fig = plt.figure(figsize=(16, 8))

for col_idx, (states, label, traj_color, title) in enumerate([
    (states_ideal, 'Ideal', 'plasma',  f'Ideal (F={compute_fidelity(sv_final_ideal):.4f})'),
    (states_noisy, 'Noisy', 'inferno', f'Noisy (F={fidelity_noisy:.4f})'),
]):
    ax = fig.add_subplot(1, 2, col_idx + 1, projection='3d')

    # Sphere wireframe
    ax.plot_wireframe(sx, sy, sz, color='lightgrey', alpha=0.18, linewidth=0.4)

    # Axis arrows + labels
    for vec, lbl, c in [([1,0,0], 'X', 'red'), ([0,1,0], 'Y', 'green'), ([0,0,1], 'Z', 'blue')]:
        ax.quiver(0,0,0,*vec, length=1.25, color=c, arrow_length_ratio=0.08, linewidth=1.5)
        ax.text(vec[0]*1.4, vec[1]*1.4, vec[2]*1.4, lbl, fontsize=10, color=c)

    # Trajectory
    bl = np.array([bloch_vector(np.array(s)) for s in states])
    bx_, by_, bz_ = bl[:,0], bl[:,1], bl[:,2]
    for i in range(len(bx_)-1):
        frac = i / len(bx_)
        ax.plot(bx_[i:i+2], by_[i:i+2], bz_[i:i+2],
                color=plt.get_cmap(traj_color)(frac), lw=2.5)

    # Start, end, target
    ax.scatter(0, 0, 1,  s=120, c='lime',   zorder=5, label='|0⟩ start')
    ax.scatter(*bl[-1],  s=120, c='magenta',zorder=5, label=f'Final state')
    ax.scatter(0, 0, -1, s=80,  c='magenta',marker='*', alpha=0.5, label='|1⟩ target')

    # Draw dashed line from final to target to visualise miss distance
    if label == 'Noisy':
        ax.plot([bl[-1][0], 0], [bl[-1][1], 0], [bl[-1][2], -1],
                'r--', lw=1.5, alpha=0.7, label=f'Miss = {np.linalg.norm(bl[-1]-[0,0,-1]):.3f}')

    ax.set_xlim([-1.3,1.3]); ax.set_ylim([-1.3,1.3]); ax.set_zlim([-1.3,1.3])
    ax.set_xlabel('X', fontsize=10); ax.set_ylabel('Y', fontsize=10); ax.set_zlabel('Z', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)

fig.suptitle("Bloch Sphere: Noise Causes the State Vector to Miss |1⟩", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('phase3_bloch_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*55}")
print(f"  PHASE 3 SUMMARY — NOISE IMPACT")
print(f"{'='*55}")
print(f"  Ideal fidelity    : {compute_fidelity(sv_final_ideal):.6f}")
print(f"  Noisy fidelity    : {fidelity_noisy:.6f}")
print(f"  Fidelity loss     : {compute_fidelity(sv_final_ideal) - fidelity_noisy:.6f}  ({(1-fidelity_noisy)*100:.2f}% error rate)")
print(f"  Bloch miss dist   : {np.linalg.norm(np.array(bloch_noisy[-1]) - np.array([0,0,-1])):.4f}")

---
## Automated Pulse Reshaping via Quantum Optimal Control

### Strategy: Classical Feedback Loop

I implement Gradient-Free Quantum Optimal Control using a classical optimiser as the outer loop:

```
┌──────────────────────────────────────────────────────────┐
│  COBYLA / Nelder-Mead Optimiser                           │
│  Parameters: [A_opt, σ_opt]                              │
│                           │                              │
│                    ┌──────▼──────┐                       │
│                    │  Cost Fn.   │  ← 1 - F(ψ_final, |1⟩)│
│                    └──────┬──────┘                       │
│                           │                              │
│              ┌────────────▼────────────┐                 │
│              │  Qiskit Dynamics Solver │  ← Noisy H(t)   │
│              └─────────────────────────┘                 │
└──────────────────────────────────────────────────────────┘
```

The optimiser cannot "see" the noise — it only receives the fidelity score. By adjusting $A$ and $\sigma$, it learns to pre-distort the pulse so that even after the noise is applied, the final state is close to $|1\rangle$.

This is the core idea behind DRAG (Derivative Removal via Adiabatic Gate) pulses and other hardware-aware optimal control schemes used in IBM Quantum and Google Sycamore processors.

In [ ]:
### Cost function

iteration_log = []     # record every evaluation for convergence plotting

def cost_function(params):
    """
    Infidelity = 1 - F(|ψ_noisy(T)⟩, |1⟩)

    Parameters
    ----------
    params : [A_candidate, sigma_candidate]
        Pulse parameters proposed by the optimiser.

    Returns
    -------
    float : infidelity ∈ [0, 1] (0 = perfect, 1 = completely wrong)
    """
    A_cand, sigma_cand = params

    # Safety bounds: reject unphysical parameters
    if sigma_cand <= 0.1 or A_cand <= 0.0:
        return 1.0 

    # Run with FIXED noise on top of the candidate pulse
    states, _ = run_noisy_simulation(A_cand, sigma_cand,
                                      amp_jitter=AMP_JITTER_FRAC,
                                      delta=DELTA_NOISE)

    sv_final   = np.array(states[-1])
    fidelity   = compute_fidelity(sv_final)
    infidelity = 1.0 - fidelity

    # Log for convergence plot
    iteration_log.append({
        'A': A_cand,
        'sigma': sigma_cand,
        'fidelity': fidelity,
        'infidelity': infidelity,
    })

    return infidelity


print("Cost function defined.")
print(f"Starting point: A={A_ideal:.4f}, σ={sigma_ideal:.4f}")
print(f"Starting infidelity (noisy): {1 - fidelity_noisy:.6f}")

In [ ]:
# Nelder-Mead optimiser

print("Starting optimisation... (this may take ~30-60 seconds)")
print(f"{'─'*50}")

x0 = [A_ideal, sigma_ideal]   

opt_result = minimize(
    cost_function,
    x0,
    method='Nelder-Mead',
    options={
        'xatol':   1e-7,       
        'fatol':   1e-8,        
        'maxiter': 600,         
        'disp':    False,
    }
)

A_opt     = opt_result.x[0]
sigma_opt = opt_result.x[1]
fid_opt   = 1.0 - opt_result.fun

print(f"{'─'*50}")
print(f"Optimisation complete in {opt_result.nit} iterations ({len(iteration_log)} cost evaluations)")
print()
print(f"{'Parameter':<20} {'Original (Ideal)':<20} {'Optimised':<20} {'Δ Change'}")
print(f"{'─'*75}")
print(f"{'Amplitude A':<20} {A_ideal:<20.6f} {A_opt:<20.6f} {A_opt - A_ideal:+.6f}")
print(f"{'Width σ':<20} {sigma_ideal:<20.6f} {sigma_opt:<20.6f} {sigma_opt - sigma_ideal:+.6f}")
print()
print(f"{'Fidelity Metric':<30} {'Value'}")
print(f"{'─'*45}")
print(f"{'Ideal (noise-free) F':<30} {compute_fidelity(sv_final_ideal):.6f}")
print(f"{'Noisy (pre-opt) F':<30} {fidelity_noisy:.6f}")
print(f"{'Optimised (post-opt) F':<30} {fid_opt:.6f}")
print(f"{'Recovery':<30} {fid_opt - fidelity_noisy:+.6f}")

In [ ]:
# Convergence plot
fidelity_history = [entry['fidelity'] for entry in iteration_log]
A_history        = [entry['A']        for entry in iteration_log]
sigma_history    = [entry['sigma']    for entry in iteration_log]
iters            = list(range(1, len(fidelity_history) + 1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Phase 4 — Optimiser Convergence History", fontsize=14, fontweight='bold')

# Fidelity convergence
ax1 = axes[0]
ax1.plot(iters, fidelity_history, color='darkorange', lw=1.5, alpha=0.7)
ax1.axhline(fid_opt,        color='green',     ls='--', lw=2, label=f'Converged F={fid_opt:.5f}')
ax1.axhline(fidelity_noisy, color='crimson',   ls=':',  lw=2, label=f'Initial F={fidelity_noisy:.5f}')
ax1.set_xlabel("Optimiser Iteration", fontsize=11)
ax1.set_ylabel("Gate Fidelity F", fontsize=11)
ax1.set_title("Fidelity vs Iteration", fontsize=11)
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# Amplitude trajectory
ax2 = axes[1]
ax2.plot(iters, A_history, color='steelblue', lw=1.5, alpha=0.7)
ax2.axhline(A_ideal, color='grey',     ls='--', lw=1.5, label=f'Ideal A={A_ideal:.4f}')
ax2.axhline(A_opt,   color='navy',     ls='-',  lw=2,   label=f'Opt.  A={A_opt:.4f}')
ax2.set_xlabel("Optimiser Iteration", fontsize=11)
ax2.set_ylabel("A [rad/ns]", fontsize=11)
ax2.set_title("Amplitude A Search Path", fontsize=11)
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

# Sigma trajectory
ax3 = axes[2]
ax3.plot(iters, sigma_history, color='purple', lw=1.5, alpha=0.7)
ax3.axhline(sigma_ideal, color='grey',   ls='--', lw=1.5, label=f'Ideal σ={sigma_ideal:.4f}')
ax3.axhline(sigma_opt,   color='indigo', ls='-',  lw=2,   label=f'Opt.  σ={sigma_opt:.4f}')
ax3.set_xlabel("Optimiser Iteration", fontsize=11)
ax3.set_ylabel("σ [ns]", fontsize=11)
ax3.set_title("Pulse Width σ Search Path", fontsize=11)
ax3.legend(fontsize=9); ax3.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('phase4_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: phase4_convergence.png")

---
Verification, Data Visualisation & Summary

With the optimised parameters in hand, we
1. Verify the optimised pulse restores high fidelity in the noisy environment
2. Compare the three pulse shapes side by side (ideal, corrupted, optimised)
3. Compare the Bloch sphere trajectories for all three scenarios
4. Quantify the fidelity recovery and summarise the experiment

In [ ]:
# Run optimised simulation through the SAME noisy environment
states_opt, A_eff_opt = run_noisy_simulation(A_opt, sigma_opt,
                                               amp_jitter=AMP_JITTER_FRAC,
                                               delta=DELTA_NOISE)

sv_final_opt = np.array(states_opt[-1])
fidelity_opt = compute_fidelity(sv_final_opt)

prob0_opt, prob1_opt, bloch_opt = extract_trajectory(states_opt)

print("Optimised pulse run through SAME noisy environment:")
print(f"  Input  A_opt    = {A_opt:.6f} rad/ns  (pre-distorted)")
print(f"  Applied A_eff   = {A_eff_opt:.6f} rad/ns  (after +{AMP_JITTER_FRAC*100:.0f}% jitter)")
print(f"  Final state     = {sv_final_opt}")
print(f"  Gate Fidelity   = {fidelity_opt:.6f}  ← should be close to 1.0")
print(f"  Improvement     = {fidelity_opt - fidelity_noisy:+.6f}")

In [ ]:
# FINAL FIGURE 1: Three Way Pulse Shape Comparison

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)
fig.suptitle("Phase 5 — Pulse Shape Comparison: Ideal vs Corrupted vs Optimised",
             fontsize=14, fontweight='bold')

time_fine = np.linspace(0, T_gate, 800)

# Compute all three envelopes
env_ideal  = A_ideal    * np.exp(-(time_fine - t0)**2 / (2 * sigma_ideal**2))
env_noisy  = A_eff_noisy * np.exp(-(time_fine - t0)**2 / (2 * sigma_ideal**2))
env_opt    = A_eff_opt   * np.exp(-(time_fine - t0)**2 / (2 * sigma_opt**2))

configs = [
    (axes[0], env_ideal,  'steelblue', f'Ideal Gaussian\nA={A_ideal:.4f}, σ={sigma_ideal:.4f}',
     f'Noiseless F = {compute_fidelity(sv_final_ideal):.5f}'),
    (axes[1], env_noisy,  'crimson',   f'Corrupted Pulse (applied)\nA={A_eff_noisy:.4f}, σ={sigma_ideal:.4f}',
     f'Noisy F = {fidelity_noisy:.5f}'),
    (axes[2], env_opt,    'seagreen',  f'Optimised Pulse (applied)\nA={A_eff_opt:.4f}, σ={sigma_opt:.4f}',
     f'Recovered F = {fidelity_opt:.5f}'),
]

for ax, env, color, label, fid_txt in configs:
    ax.fill_between(time_fine, env, alpha=0.2, color=color)
    ax.plot(time_fine, env, color=color, lw=2.8, label=label)
    ax.axhline(0, color='grey', lw=0.8, ls='--', alpha=0.5)

    peak = env.max()
    ax.annotate(f'Peak = {peak:.4f}', xy=(t0, peak), xytext=(t0 + T_gate*0.2, peak * 0.8),
                fontsize=9, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=1.2))

    ax.set_xlabel("Time [ns]", fontsize=11)
    ax.set_ylabel("Ω(t) [rad/ns]", fontsize=11)
    ax.set_title(label, fontsize=11, color=color, fontweight='bold')
    ax.grid(alpha=0.3)

    # Fidelity badge
    fid_val = float(fid_txt.split('=')[1])
    badge_color = 'honeydew' if fid_val > 0.98 else '#fff0f0'
    border_color = 'green' if fid_val > 0.98 else 'crimson'
    ax.text(0.05, 0.92, fid_txt, transform=ax.transAxes, fontsize=10,
            color='darkgreen' if fid_val > 0.98 else 'crimson',
            bbox=dict(boxstyle='round,pad=0.35', facecolor=badge_color, edgecolor=border_color))

# Overlay ideal dashed line on optimised for visual comparison
axes[2].plot(time_fine, env_ideal, color='steelblue', lw=1.5, ls='--', alpha=0.5, label='Ideal (reference)')
axes[2].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('phase5_pulse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: phase5_pulse_comparison.png")

In [ ]:
# FINAL FIGURE 2: Probability Trajectory Comparison (all 3 scenarios)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Phase 5 — Gate Trajectory Recovery: Fidelity Before vs After Optimisation",
             fontsize=14, fontweight='bold')

# Left: P(|1⟩) for all three runs
ax1 = axes[0]
ax1.plot(t_eval, prob1_ideal, color='steelblue', lw=2.5, ls='--',
         label=f'Ideal (noise-free)   F={compute_fidelity(sv_final_ideal):.5f}')
ax1.plot(t_eval, prob1_noisy, color='crimson',   lw=2.5,
         label=f'Noisy (pre-opt)      F={fidelity_noisy:.5f}')
ax1.plot(t_eval, prob1_opt,   color='seagreen',  lw=2.5,
         label=f'Optimised (post-opt) F={fidelity_opt:.5f}')

ax1.axhline(1.0, color='grey', lw=0.8, ls=':', alpha=0.6)
ax1.fill_between(t_eval, prob1_noisy, prob1_opt, alpha=0.15, color='seagreen',
                 label='Recovered region')
ax1.set_xlabel("Time [ns]", fontsize=12)
ax1.set_ylabel("P(|1⟩) — Target State Population", fontsize=12)
ax1.set_title("State Population P(|1⟩) — All Three Runs", fontsize=12)
ax1.set_ylim(-0.05, 1.08)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Right: fidelity bar chart for clear comparison
ax2 = axes[1]
scenarios = ['Ideal\n(noise-free)', 'Noisy\n(pre-opt)', 'Optimised\n(post-opt)']
fidelities = [compute_fidelity(sv_final_ideal), fidelity_noisy, fidelity_opt]
colors_bar  = ['steelblue', 'crimson', 'seagreen']
bars = ax2.bar(scenarios, fidelities, color=colors_bar, edgecolor='black', linewidth=1.2,
               width=0.5, alpha=0.85)

for bar, fid in zip(bars, fidelities):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{fid:.5f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax2.axhline(0.99, color='gold', ls='--', lw=2, label='99% threshold')
ax2.set_ylabel("Gate Fidelity F", fontsize=12)
ax2.set_ylim(0.85, 1.04)
ax2.set_title("Gate Fidelity: Three-Way Comparison", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('phase5_fidelity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: phase5_fidelity_comparison.png")

In [ ]:
# FINAL FIGURE 3: Bloch Sphere — Three-Way Trajectory Comparison

fig = plt.figure(figsize=(20, 7))
fig.suptitle("Phase 5 — Bloch Sphere: Ideal → Corrupted → Recovered",
             fontsize=15, fontweight='bold', y=0.98)

scenario_data = [
    (states_ideal, 'plasma',   f'1. Ideal Gate\nF = {compute_fidelity(sv_final_ideal):.5f}'),
    (states_noisy, 'inferno',  f'2. Noisy (Pre-Opt)\nF = {fidelity_noisy:.5f}'),
    (states_opt,   'viridis',  f'3. Optimised (Post-Opt)\nF = {fidelity_opt:.5f}'),
]

for idx, (states, cmap_name, title_str) in enumerate(scenario_data):
    ax = fig.add_subplot(1, 3, idx + 1, projection='3d')

    # Sphere
    ax.plot_wireframe(sx, sy, sz, color='lightgrey', alpha=0.18, linewidth=0.4)

    # Reference axes
    for vec, lbl, c in [([1,0,0],'X','red'), ([0,1,0],'Y','green'), ([0,0,1],'Z','blue')]:
        ax.quiver(0,0,0,*vec, length=1.2, color=c, arrow_length_ratio=0.1, linewidth=1.5)
        ax.text(vec[0]*1.35, vec[1]*1.35, vec[2]*1.35, lbl, fontsize=10, color=c)

    # Label poles
    ax.text(0, 0, 1.15, '|0⟩', fontsize=11, ha='center', color='navy')
    ax.text(0, 0, -1.2, '|1⟩', fontsize=11, ha='center', color='darkmagenta')

    # Trajectory
    bl = np.array([bloch_vector(np.array(s)) for s in states])
    bx_, by_, bz_ = bl[:,0], bl[:,1], bl[:,2]
    for i in range(len(bx_)-1):
        frac = i / max(len(bx_)-1, 1)
        ax.plot(bx_[i:i+2], by_[i:i+2], bz_[i:i+2],
                color=plt.get_cmap(cmap_name)(frac), lw=2.8)

    # Start / end / target
    ax.scatter(0, 0,  1,  s=150, c='lime',    zorder=6, marker='o', label='Start |0⟩')
    ax.scatter(*bl[-1],   s=150, c='magenta', zorder=6, marker='o', label='Final |ψ(T)⟩')
    ax.scatter(0, 0, -1,  s=100, c='magenta', marker='*', alpha=0.6, label='Target |1⟩')

    # Show geodesic miss for noisy
    if idx == 1:
        ax.plot([bl[-1][0], 0], [bl[-1][1], 0], [bl[-1][2], -1],
                'r--', lw=2, alpha=0.8)
        miss = np.linalg.norm(bl[-1] - np.array([0,0,-1]))
        ax.text(-0.2, 0, -0.5, f'miss={miss:.3f}', fontsize=9, color='red')

    ax.set_xlim([-1.3,1.3]); ax.set_ylim([-1.3,1.3]); ax.set_zlim([-1.3,1.3])
    ax.set_xlabel('X',fontsize=10); ax.set_ylabel('Y',fontsize=10); ax.set_zlabel('Z',fontsize=10)
    ax.set_title(title_str, fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.savefig('phase5_bloch_all_three.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: phase5_bloch_all_three.png")

In [ ]:
# Final Summary Table & Metrics

target_sv = np.array([0.0, 1.0], dtype=complex)

bloch_final_ideal = bloch_vector(sv_final_ideal)
bloch_final_noisy = bloch_vector(sv_final_noisy)
bloch_final_opt   = bloch_vector(sv_final_opt)

miss_ideal = np.linalg.norm(np.array(bloch_final_ideal) - np.array([0, 0, -1]))
miss_noisy = np.linalg.norm(np.array(bloch_final_noisy) - np.array([0, 0, -1]))
miss_opt   = np.linalg.norm(np.array(bloch_final_opt)   - np.array([0, 0, -1]))

print("\n" + "═"*70)
print("  QUANTUM PULSE OPTIMAL CONTROL — EXPERIMENT SUMMARY")
print("═"*70)
print()
print(f"  NOISE MODEL")
print(f"    Amplitude jitter    : +{AMP_JITTER_FRAC*100:.0f}% systematic over-drive")
print(f"    Detuning drift      : δ = {DELTA_NOISE:.4f} rad/ns  (H_drift = δ/2 · Z)")
print()
print(f"{'─'*70}")
print(f"  {'Scenario':<25} {'Fidelity':>10} {'Infidelity':>12} {'Bloch Miss':>12}")
print(f"{'─'*70}")

rows = [
    ('Ideal (no noise)',      compute_fidelity(sv_final_ideal), miss_ideal),
    ('Noisy (pre-opt)',       fidelity_noisy,                   miss_noisy),
    ('Optimised (post-opt)',  fidelity_opt,                     miss_opt),
]
for name, fid, miss in rows:
    inf = 1 - fid
    marker = ' ◄ RECOVERED' if name.startswith('Opt') else ''
    print(f"  {name:<25} {fid:>10.6f} {inf:>12.6f} {miss:>12.6f}{marker}")

print(f"{'─'*70}")
print()
print(f"  OPTIMISER PERFORMANCE")
print(f"    Method           : Nelder-Mead (derivative-free simplex)")
print(f"    Evaluations      : {len(iteration_log)}")
print(f"    Iterations       : {opt_result.nit}")
print(f"    Convergence      : {'✓ Yes' if opt_result.success else '✗ No'} — {opt_result.message}")
print()
print(f"  OPTIMISED PULSE PARAMETERS")
print(f"    Pre-distorted A  : {A_opt:.6f} rad/ns   (ideal was {A_ideal:.6f})")
print(f"    Pre-distorted σ  : {sigma_opt:.6f} ns     (ideal was {sigma_ideal:.6f})")
print(f"    Effective A (%)  : {(A_opt/A_ideal - 1)*100:+.2f}% offset from ideal")
print(f"    Effective σ (%)  : {(sigma_opt/sigma_ideal - 1)*100:+.2f}% offset from ideal")
print()
print(f"  KEY RESULT")
recovery_pct = (fidelity_opt - fidelity_noisy) / (1 - fidelity_noisy) * 100
print(f"    Fidelity recovered from {fidelity_noisy:.4f} → {fidelity_opt:.4f}")
print(f"    Error rate reduced: {(1-fidelity_noisy)*100:.2f}% → {(1-fidelity_opt)*100:.2f}%")
print(f"    Error recovery    : {recovery_pct:.1f}% of the noise-induced error eliminated")
print()
print("═"*70)

---
## Experiment Summary & Physics Interpretation

### What I Built
A complete, physics-grounded simulation of quantum pulse optimal control for a superconducting transmon qubit, demonstrating how a classical optimisation loop can systematically recover gate fidelity lost to hardware noise.

---

### Results At a Glance

| Scenario | Gate Fidelity | Error Rate | Bloch Miss |
|---|---|---|---|
| Ideal (noiseless) | ≈ 1.000000 | ≈ 0.000% | ≈ 0.000 |
| Noisy (pre-optimisation) | ≈ 0.918 | ≈ 8.2% | ≈ 0.26 |
| Optimised (post-optimisation) | ≈ 0.995 | ≈ 0.5% | ≈ 0.05|


> Fidelity recovered from ~0.92 → ~0.995, eliminating over 93% of the noise-induced error.

---

### Physics of the Noise Sources

1. Amplitude Jitter (+15% systematic over-drive)  
The AWG outputs $A_{\text{applied}} = A_{\text{intended}} \times 1.15$, rotating the qubit beyond the south pole. The Bloch vector over-rotates past $|1\rangle$, landing at a mixed angle.

2. Detuning Drift ($\delta = 0.05$ rad/ns)
The residual term $\frac{\delta}{2}Z$ generates an unwanted Z-axis precession concurrent with the X-rotation. This tilts the great-circle arc away from the ideal $|0\rangle \to |1\rangle$ meridian.

Together, these create a compound geometric error that shifts the trajectory off the intended Bloch sphere path.

---

### How the Optimiser Fixed It

The Nelder-Mead simplex optimiser adjusted two pulse parameters:

- Reduced amplitude $A_{\text{opt}} < A_{\text{ideal}}$: Pre-compensates for the +15% jitter, so the *applied* amplitude matches the area theorem requirement.
- Adjusted width $\sigma_{\text{opt}}$: Reshapes the temporal profile so the pulse spends more time in the regime that counteracts the Z-drift.

This is mathematically equivalent to DRAG pulse correction, which is the standard technique used on IBM Quantum, Google Sycamore, and Rigetti hardware to suppress leakage and detuning errors.